# Piloto Semana 4 — Baseline: Regresión Logística (P06)

Notebook modular: entrena **solo** el baseline sobre la partición piloto ya generada por
`01_Adquisicion_EDA.ipynb` (sección 8.1-8.2 → `dataset_split_piloto.csv`). No recalcula
adquisición ni EDA — si ese archivo no existe, correr primero `01_Adquisicion_EDA.ipynb`.

Referencia: `DOCUMENTOS ELABORADOS DE LA TESIS/Protocolo de Investigacion v2.1.md`, §6 y
§11 P06. Esta es una ejecución **piloto, no oficial** — ver la nota metodológica en la
sección 8 de `01_Adquisicion_EDA.ipynb` (muestra reducida, no la partición congelada de
Fase 3).

**Evidencia que produce:** `Pilotaje/<entorno>/evidencias/baseline_regresion_logistica/baseline_config_piloto.json`
y `metricas_baseline_piloto.json`.


In [1]:
import csv
import json
import platform
import time
from pathlib import Path

import pandas as pd
import psutil
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

SEED = 42
TARGET_COL = "Target"
NOMBRE_MODELO = "baseline_regresion_logistica"


## 0. Entorno y rutas (mismo detector que el resto del piloto)

In [2]:
_entorno_detectado = {
    "Linux": "laptop_arch_linux",
    "Windows": "escritorio_windows10",
}.get(platform.system(), "entorno_no_reconocido")

RUTA_EVIDENCIAS_PILOTO = Path(f"../../Pilotaje/{_entorno_detectado}/evidencias")
RUTA_MODELO = RUTA_EVIDENCIAS_PILOTO / NOMBRE_MODELO
RUTA_MODELO.mkdir(parents=True, exist_ok=True)
RUTA_BITACORA = Path(f"../../Pilotaje/{_entorno_detectado}/bitacora_ejecucion.csv")

_proceso = psutil.Process()


def _siguiente_contador():
    """Continúa la numeración P-01, P-02... donde la haya dejado el último notebook
    que escribió en esta bitácora (01, 02 o 03), para que los pasos de los distintos
    notebooks del piloto no se pisen con el mismo id."""
    if RUTA_BITACORA.exists() and RUTA_BITACORA.stat().st_size > 0:
        with open(RUTA_BITACORA, newline="", encoding="utf-8") as f:
            ids = [fila["id"] for fila in csv.DictReader(f)]
        numeros = [int(i.split("-")[1]) for i in ids if i.startswith("P-")]
        return max(numeros) if numeros else 0
    return 0


_contador_pasos = [_siguiente_contador()]


class registrar_paso:
    """Igual que en 01_Adquisicion_EDA.ipynb: mide tiempo y memoria (RSS, vía psutil,
    cross-platform) de un bloque y lo agrega a la bitácora compartida del entorno."""

    def __init__(self, accion, entrada, esperado, evidencia=""):
        self.accion, self.entrada, self.esperado, self.evidencia = accion, entrada, esperado, evidencia

    def __enter__(self):
        self.t0 = time.perf_counter()
        self.mem_inicio_mb = _proceso.memory_info().rss / 1024**2
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        _contador_pasos[0] += 1
        duracion = round(time.perf_counter() - self.t0, 3)
        mem_fin_mb = _proceso.memory_info().rss / 1024**2
        delta_mem_mb = round(mem_fin_mb - self.mem_inicio_mb, 1)
        observado = "OK" if exc_type is None else f"ERROR: {exc_val}"
        incidencia = "" if exc_type is None else str(exc_val)
        fila = {
            "id": f"P-{_contador_pasos[0]:02d}",
            "accion": self.accion,
            "entrada": self.entrada,
            "esperado": self.esperado,
            "observado": f"{observado} | tiempo={duracion}s | rss_actual={round(mem_fin_mb, 1)}MB | delta_mem={delta_mem_mb}MB",
            "incidencia": incidencia,
            "evidencia": self.evidencia,
        }
        existe_con_contenido = RUTA_BITACORA.exists() and RUTA_BITACORA.stat().st_size > 0
        with open(RUTA_BITACORA, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(fila.keys()))
            if not existe_con_contenido:
                w.writeheader()
            w.writerow(fila)
        print(f"[{fila['id']}] {self.accion} -> {fila['observado']}")
        return False


print(f"Entorno detectado: {_entorno_detectado}")
print(f"Evidencia de este modelo: {RUTA_MODELO}")


Entorno detectado: laptop_arch_linux
Evidencia de este modelo: ../../Pilotaje/laptop_arch_linux/evidencias/baseline_regresion_logistica


## 1. Cargar la partición piloto

Generada por `01_Adquisicion_EDA.ipynb` (sección 8.1-8.2) — **no** la partición oficial
de Fase 3.

In [3]:
df_split_piloto = pd.read_csv(RUTA_EVIDENCIAS_PILOTO / "dataset_split_piloto.csv")

cols_features_piloto = [c for c in df_split_piloto.columns if c not in {TARGET_COL, "class_binaria", "split"}]

df_train_piloto = df_split_piloto[df_split_piloto["split"] == "train"]
df_val_piloto = df_split_piloto[df_split_piloto["split"] == "val"]

X_train_piloto = df_train_piloto[cols_features_piloto]
y_train_piloto = df_train_piloto["class_binaria"]
X_val_piloto = df_val_piloto[cols_features_piloto]
y_val_piloto = df_val_piloto["class_binaria"]

print(f"train={len(X_train_piloto)}  val={len(X_val_piloto)}  variables={len(cols_features_piloto)}")


train=840  val=180  variables=36


## 2. Preprocesamiento (P05, igual que en 01): escalado + SMOTE solo en train

Se reajusta aquí (StandardScaler/SMOTE son deterministas dados los mismos datos y
`random_state`) en vez de cargar un objeto serializado — mantiene este notebook
autocontenido, sin depender de pickles de scikit-learn entre notebooks.

In [4]:
with registrar_paso(
    accion="Reajustar StandardScaler + SMOTE sobre train (mismo criterio que 01, sección 8.3)",
    entrada=f"train piloto ({len(X_train_piloto)} filas)",
    esperado="mismo resultado que en 01 (determinista, mismo seed)",
    evidencia="(no genera archivo nuevo; pipeline_config_piloto.json ya existe, generado por 01)",
):
    escalador = StandardScaler()
    X_train_escalado = escalador.fit_transform(X_train_piloto)
    X_train_proc, y_train_proc = SMOTE(random_state=SEED).fit_resample(X_train_escalado, y_train_piloto)
    X_val_proc = escalador.transform(X_val_piloto)

print(f"Train antes de SMOTE: {len(X_train_piloto)} -> después: {len(X_train_proc)}")


[P-05] Reajustar StandardScaler + SMOTE sobre train (mismo criterio que 01, sección 8.3) -> OK | tiempo=0.033s | rss_actual=208.4MB | delta_mem=3.6MB
Train antes de SMOTE: 840 -> después: 1022


## 3. Entrenar baseline (P06) y métricas sobre validación

In [5]:
with registrar_paso(
    accion="Entrenar baseline Regresión Logística (con SMOTE) y predecir sobre validación",
    entrada=f"train procesado ({len(X_train_proc)} filas tras SMOTE), val ({len(X_val_proc)} filas)",
    esperado="el modelo entrena sin error y produce probabilidades/predicciones sobre val",
    evidencia=f"{NOMBRE_MODELO}/baseline_config_piloto.json, {NOMBRE_MODELO}/metricas_baseline_piloto.json",
):
    modelo_baseline_piloto = LogisticRegression(max_iter=1000, random_state=SEED)
    modelo_baseline_piloto.fit(X_train_proc, y_train_proc)

    y_val_proba = modelo_baseline_piloto.predict_proba(X_val_proc)[:, 1]
    y_val_pred = modelo_baseline_piloto.predict(X_val_proc)

    metricas_baseline_piloto = {
        "auc_roc": round(float(roc_auc_score(y_val_piloto, y_val_proba)), 4),
        "f1": round(float(f1_score(y_val_piloto, y_val_pred)), 4),
        "precision": round(float(precision_score(y_val_piloto, y_val_pred)), 4),
        "recall": round(float(recall_score(y_val_piloto, y_val_pred)), 4),
        "exactitud": round(float(accuracy_score(y_val_piloto, y_val_pred)), 4),
        "n_val": int(len(y_val_piloto)),
    }

    baseline_config_piloto = {
        "modelo": "LogisticRegression",
        "hiperparametros": {"max_iter": 1000, "random_state": SEED},
        "smote": True,
        "seed": SEED,
    }
    with open(RUTA_MODELO / "baseline_config_piloto.json", "w", encoding="utf-8") as f:
        json.dump(baseline_config_piloto, f, indent=2, ensure_ascii=False)
    with open(RUTA_MODELO / "metricas_baseline_piloto.json", "w", encoding="utf-8") as f:
        json.dump(metricas_baseline_piloto, f, indent=2, ensure_ascii=False)

print(metricas_baseline_piloto)


[P-06] Entrenar baseline Regresión Logística (con SMOTE) y predecir sobre validación -> OK | tiempo=0.02s | rss_actual=209.3MB | delta_mem=1.0MB
{'auc_roc': 0.9633, 'f1': 0.8652, 'precision': 0.8714, 'recall': 0.8592, 'exactitud': 0.8944, 'n_val': 180}
